

### Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---

**🔐 Як безпечно зберігати і підвантажувати API-ключі**

API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.



In [98]:

import json
import uuid
from typing import Annotated, TypedDict
from langchain_experimental.utilities import PythonREPL
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import Tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_classic import hub
from langchain_classic.agents import Tool, AgentExecutor, create_react_agent
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

In [ ]:
with open('creds.json') as file:
    creds = json.load(file)
api_key = creds['GEMINI_API_KEY']

In [ ]:
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=api_key,
    temperature=0.2
)

request = (
    'Поясни що таке квантові обчислення простою мовою. '
    'Напиши визначення, переваги та поточні дослідження. '
    'Відповідь має бути дуже короткою, до 200 символів.'
)

response = llm.invoke(request)
print(response.content)

Квантові обчислення використовують квантові явища для надшвидких розрахунків. Переваги: вирішення складних задач (ліки, матеріали, ШІ). Дослідження: на етапі розробки прототипів, ще не для масового використання.


### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [22]:
topics = [
    "Баєсівські методи в машинному навчанні",
    "Трансформери в машинному навчанні",
    "Explainable AI"
]

In [ ]:
template_text = (
    'Поясни що таке {topic}, простою мовою. '
    'Напиши визначення, переваги та поточні дослідження. '
    'Відповідь має бути дуже короткою, до 300 символів.'
)

prompt = PromptTemplate(
    input_variables=['topic'],
    template=template_text
)
chain = prompt | llm

In [29]:
for t in topics:
    print(f'----Тема: {t}----\n')
    result = chain.invoke({'topic': t})
    print(result.content)
    print('\n')

----Тема: Баєсівські методи в машинному навчанні----

Баєсівські методи оновлюють наші переконання (імовірності) за допомогою нових даних.

**Переваги**: врахування невизначеності, ефективність з малими даними.

**Дослідження**: масштабованість, інтеграція з глибоким навчанням.


----Тема: Трансформери в машинному навчанні----

Трансформери – це нейромережі для послідовностей, що використовують механізм уваги для паралельної обробки та кращого розуміння довгих залежностей. Переваги: швидкість, ефективність. Дослідження: масштабування, мультимодальність.


----Тема: Explainable AI----

**Explainable AI (XAI)** робить рішення ШІ зрозумілими, пояснюючи "чому". Це підвищує довіру, прозорість та допомагає виявляти помилки. Дослідження зосереджені на нових методах пояснення та їх оцінці.






### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

"2 of 2. Barack Obama: timeline Key events in the life of Barack Obama. Barack Obama (born August 4, 1961, Honolulu, Hawaii, U.S.) is the 44th president of the United States (2009-17) and the first African American to hold the office. Before winning the presidency, Obama represented Illinois in the U.S. Senate (2005-08). Since the office was established in 1789, 45 men have served in 46 presidencies. The first president, George Washington, won a unanimous vote of the Electoral College. [4] Grover Cleveland served two non-consecutive terms and is therefore counted as the 22nd and 24th president of the United States, giving rise to the discrepancy between the ... Here is a list of the presidents and vice presidents of the United States along with their parties and dates in office. ... Chester A Arthur: Twenty-First President of the United States. 10 Interesting Facts About James Buchanan. Martin Van Buren - Eighth President of the United States. Quotes From Harry S. Truman. Table of Cont

In [66]:
search = DuckDuckGoSearchRun()
prompt = hub.pull('hwchase17/react')
tools = [
    Tool(
        name='Search',
        func=search.run,
        description='Використовуй для пошуку актуальної інформації в інтернеті.'
    )
]
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,        
    handle_parsing_errors=True,
    early_stopping_method='force',
    verbose=True
)

agent_executor.invoke(
    {'input': 'Знайди 5 останніх публікацій на тему штучного інтелекту. Напиши назву, авторів та короткий опис до 200 символів.'}
)



> Entering new AgentExecutor chain...
Action: Search
Action Input: latest 5 publications on artificial intelligence research with authors and short description3 weeks ago -Subjects: Artificial Intelligence (cs.AI); Human-Computer Interaction (cs.HC) ... Title: ProMedical: Hierarchical Fine-Grained Criteria Modeling for Medical LLM Alignment via Explicit Injection · He Geng, Yangmin Huang, Lixian Lai, Qianyun Du, Hui Chu, Zhiyang He, Jiaxue Hu, Xiaodong Tao ... Comments: 5 pages, 3 figures. 2 hours ago -16, 2025 Electrons can freeze into strange geometric crystals and then melt back into liquid-like motion under the right quantum conditions. Researchers identified how to tune these transitions and even ... Artificial Neurons That Behave Like Real Brain Cells · Nov. 5, 2025 USC researchers built artificial neurons that replicate real brain processes using ion-based diffusive memristors. 4 days ago -Artificial Intelligence Reviewis a fully open access journal publishing state-of-the-art

{'input': 'Знайди 5 останніх публікацій на тему штучного інтелекту. Напиши назву, авторів та короткий опис до 200 символів.',
 'output': 'Ось 5 останніх публікацій на тему штучного інтелекту, зібраних на основі доступної інформації:\n\n1.  **Назва:** ProMedical: Hierarchical Fine-Grained Criteria Modeling for Medical LLM Alignment via Explicit Injection\n    **Автори:** He Geng, Yangmin Huang, Lixian Lai, Qianyun Du, Hui Chu, Zhiyang He, Jiaxue Hu, Xiaodong Tao\n    **Опис:** Дослідження пропонує ієрархічне моделювання критеріїв для покращення вирівнювання великих мовних моделей (LLM) у медичній сфері, підвищуючи їх точність.\n\n2.  **Назва:** Euclidean fast attention: a linear-scaling framework for 3D data\n    **Автори:** J. Thorben Frank, Oliver T. Unke та ін.\n    **Опис:** Представляє лінійно-масштабовану структуру для 3D-даних, яка використовує евклідові обертові кодування для ефективного захоплення довгострокових ефектів.\n\n3.  **Назва:** Alignment of Artificial Neural Network 



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [ ]:
search = DuckDuckGoSearchRun()
python_repl = PythonREPL()

tools = [
    Tool(
        name='Search',
        func=search.run,
        description='Використовуй коли тобі потрібно знайти актуальні дані в інтернеті, прогнози погоди або економічні новини.'
    ),
    Tool(
        name='Python_REPL',
        func=python_repl.run,
        description='Використовуй для написання та виконання коду Python, для математичних розрахунків, прогнозів та аналізу трендів.'
    )
]

prompt = hub.pull('hwchase17/react')
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,        
    handle_parsing_errors=True,
    early_stopping_method='force',
    verbose=True
)

user_request = """
Ми експортуємо апельсини з Бразилії. 
В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. 
Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії 
і попит на апельсини в світі виходячи з економічної ситуації.
Видай результат бізнес-аналізу або напиши, що не можеш цього зробити.
"""

agent_executor.invoke({'input': user_request})



> Entering new AgentExecutor chain...


Python REPL can execute arbitrary code. Use with caution.


Thought: Мені потрібно спрогнозувати експорт апельсинів з Бразилії на 2026 рік, враховуючи історичні дані, погодні умови в Бразилії та світовий попит.

Кроки:
1.  Проаналізувати надані історичні дані експорту за допомогою `Python_REPL` для виявлення тренду.
2.  Використати `Search` для пошуку інформації про довгострокові прогнози погоди в Бразилії, які можуть вплинути на врожай апельсинів у 2026 році.
3.  Використати `Search` для пошуку інформації про світову економічну ситуацію та попит на апельсини.
4.  На основі зібраної інформації зробити оцінку або зазначити, що це неможливо.

Почнемо з аналізу історичних даних.
Action: Python_REPL
Action Input:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

years = np.array([2022, 2023, 2024, 2025]).reshape(-1, 1)
exports = np.array([200, 190, 210, 220])

model = LinearRegression()
model.fit(years, exports)

# Predict for 2026
year_2026 = np.array([[2026]])
predicted_2026 = model.predict(year_2026)
print

{'input': '\nМи експортуємо апельсини з Бразилії. \nВ 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. \nЗроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії \nі попит на апельсини в світі виходячи з економічної ситуації.\nВидай результат бізнес-аналізу або напиши, що не можеш цього зробити.\n',
 'output': 'Я не можу зробити повний бізнес-аналіз та надати точну оцінку експорту апельсинів на 2026 рік, яка б враховувала всі запитувані фактори (погодні умови в Бразилії та світовий попит виходячи з економічної ситуації).\n\nНа основі лише історичних даних експорту (2022: 200т, 2023: 190т, 2024: 210т, 2025: 220т), лінійний тренд прогнозує експорт у 2026 році на рівні **225 тонн**.\n\nОднак, ця оцінка не враховує зовнішні фактори, які були запитані, і є лише базовим прогнозом. Існує висока ймовірність того, що фактичний експорт у 2026 році може бути значно нижчим через поширення хвороби зелянення цитрусових у Бразилії, яка заг

In [ ]:
search = DuckDuckGoSearchRun()
python_repl = PythonREPL()

tools = [
    Tool(
        name='Search',
        func=search.run,
        description='Використовуй для пошуку актуальної інформації в інтернеті.'
    ),
    Tool(
        name='Python_REPL',
        func=python_repl.run,
        description='Використовуй для математичних розрахунків та прогнозів.'
    )
]

In [96]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def call_model(state: State):
    messages = state['messages']
    response = llm.bind_tools(tools).invoke(messages)
    return {'messages': [response]}

def tool_node(state: State):
    messages = state['messages']
    last_message = messages[-1]
    results = []
    
    for tool_call in last_message.tool_calls:
        tool = {t.name: t for t in tools}[tool_call['name']]
        observation = tool.invoke(tool_call['args'])
        # Створюємо правильний об'єкт ToolMessage для LangGraph
        results.append(ToolMessage(
            tool_call_id=tool_call['id'],
            content=str(observation)
        ))
    return {'messages': results}

def should_continue(state: State):
    last_message = state['messages'][-1]
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return 'tools'
    return 'end'

In [97]:
workflow = StateGraph(State)
workflow.add_node('agent', call_model)
workflow.add_node('tools', tool_node)

workflow.set_entry_point('agent')
workflow.add_conditional_edges('agent', should_continue, {'tools': 'tools', 'end': END})
workflow.add_edge('tools', 'agent')

app = workflow.compile()

In [95]:
user_input = '''
Ми експортуємо апельсини з Бразилії. 
В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. 
Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії 
і попит на апельсини в світі виходячи з економічної ситуації.
'''

initial_state = {'messages': [HumanMessage(content=user_input)]}

for chunk in app.stream(initial_state):
    for node_name, output in chunk.items():
        print(f'Node: {node_name}')
        last_msg = output['messages'][-1]
        
        if hasattr(last_msg, 'content'):
            print(last_msg.content)
        
        if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
            for tc in last_msg.tool_calls:
                print(f'Виклик інструменту: {tc['name']}({tc['args']})')

Node: agent

Виклик інструменту: Search({'Search': 'прогноз погоди в Бразилії для вирощування апельсинів 2025-2026'})
Виклик інструменту: Search({'Search': 'прогноз світового попиту на апельсини 2026 економічна ситуація'})
Node: tools
Економічніпрогнозита ризики для росту. Економісти прогнозують, що у2026році зростаннясвітовоїекономіки залишатиметься помірним, без очікуваного раптового прискорення. Основні ризики пов’язані з нерівномірністю споживчогопопиту... Апрель обещает стать временем тонких душевных резонансов и неожиданных поворотов в личной жизни. Узнаем, что приготовил для нас любовный гороскопна11 апреля2026года. Подробно о погодена10 дней в Омске и других городах России и мира. Мы покажем, как в Омске будут изменяться температура воздуха, облачность, осадки, давление, влажность, а также проинформируем вас о времени восхода и захода Солнца и фазах Луны. Интерактивная карта Украины с городами онлайн, схемы дорог и маршруты общественного транспорта. Карты Украины для скачивания

#### За результатами тестування обох архітектур (AgentExecutor та LangGraph) можна зробити висновок, що обидва підходи успішно впоралися з поставленим завданням, об’єднавши історичні дані з актуальним пошуком.

#### У процесі роботи було помічено, що агент іноді потребує уточнення пошукових запитів. Наприклад, додавання назв організацій (як USDA) або специфічних термінів (цитрусовий пояс Сан-Паулу) дозволяє отримати професійну статистику замість побутових новин. В майбутньому можна інтегрувати в систему базу знань (RAG) з офіційними звітами аграрних відомств для миттєвого доступу до точних цифр без необхідності фільтрувати результати загального пошуку.